# WLD-USDT Order Book Analysis with Database Storage
This notebook fetches and analyzes order book data from Binance Perpetual for the WLD-USDT trading pair, with database integration for backtesting.

In [1]:
import asyncio
import logging
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from decimal import Decimal
import sys
import os

# Add the project root to the path so we can import our modules
sys.path.append(os.path.abspath('../..'))

from core.data_sources.clob import CLOBDataSource

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Initialize CLOBDataSource
First, we'll initialize the CLOBDataSource which provides access to the exchange APIs.

In [2]:
# Initialize CLOBDataSource
clob = CLOBDataSource()
print("CLOBDataSource initialized successfully.")

CLOBDataSource initialized successfully.


## Database Setup
Let's set up the database connection for persistent storage of order book data.

In [3]:
# Install required libraries if not already installed
try:
    import sqlalchemy
    import psycopg2
except ImportError:
    print("Installing required database libraries...")
    !pip install sqlalchemy psycopg2-binary



In [4]:
# Database configuration
DB_CONFIG = {
    'host': 'localhost',
    'port': 5438,
    'user': 'backtest_user',
    'password': 'backtest_password',
    'database': 'backtest_db'
}

# Create SQLAlchemy engine and tables
from sqlalchemy import create_engine, Table, Column, Integer, Float, String, DateTime, MetaData
import datetime

# Create connection string
connection_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

try:
    # Create engine
    engine = create_engine(connection_string)
    print("Database engine created successfully.")
    
    # Create metadata and tables if they don't exist
    metadata = MetaData()
    
    # Define the order book bids table
    bids_table = Table(
        'order_book_bids', metadata,
        Column('id', Integer, primary_key=True),
        Column('timestamp', DateTime),
        Column('connector', String),
        Column('trading_pair', String),
        Column('price', Float),
        Column('amount', Float),
        Column('value', Float)
    )
    
    # Define the order book asks table
    asks_table = Table(
        'order_book_asks', metadata,
        Column('id', Integer, primary_key=True),
        Column('timestamp', DateTime),
        Column('connector', String),
        Column('trading_pair', String),
        Column('price', Float),
        Column('amount', Float),
        Column('value', Float)
    )
    
    # Define the order book summary table
    summary_table = Table(
        'order_book_summary', metadata,
        Column('id', Integer, primary_key=True),
        Column('timestamp', DateTime),
        Column('connector', String),
        Column('trading_pair', String),
        Column('bid_price', Float),
        Column('ask_price', Float),
        Column('spread', Float),
        Column('mid_price', Float),
        Column('imbalance', Float)
    )
    
    # Create the tables in the database
    metadata.create_all(engine)
    print("Database tables created or already exist.")
    
except Exception as e:
    print(f"Error connecting to database: {e}")
    print("Continuing without database functionality.")
    engine = None

Database engine created successfully.
Database tables created or already exist.


## Define Order Book Functions
Since the order book snapshot method is not yet implemented in CLOBDataSource (there's a TODO comment), we'll implement it here directly using the connector.



In [5]:
async def get_order_book(trading_pair, limit=500):
    """
    Get order book data for a specific trading pair.
    
    Args:
        trading_pair: Trading pair in internal format (e.g., 'WLD-USDT')
        limit: Order book depth (default: 500)
        
    Returns:
        Dictionary containing order book data
    """
    clob = CLOBDataSource()
    connector = clob.connectors.get("binance_perpetual")
    
    # Convert from internal format (WLD-USDT) to exchange format (WLDUSDT)
    symbol = trading_pair.replace('-', '')
    
    try:
        # Make API request to get order book
        response = await connector._api_get(
            path_url="/fapi/v1/depth",
            params={"symbol": symbol, "limit": limit},
            is_auth_required=False,
            limit_id="REQUEST_WEIGHT"
        )
        
        # Process data into a more usable format
        order_book = {
            "timestamp": datetime.now(),
            "last_update_id": response["lastUpdateId"],
            "bids": pd.DataFrame(response["bids"], columns=["price", "quantity"], dtype=float),
            "asks": pd.DataFrame(response["asks"], columns=["price", "quantity"], dtype=float)
        }
        
        return order_book
    except Exception as e:
        print(f"Failed to fetch order book: {e}")
        return None

## Fetch Order Book Data
Now we'll fetch the order book data for WLD-USDT from Binance Perpetual and save it to the database.

In [6]:
def plot_order_book(order_book, levels=10):
    """
    Plot the order book with specified number of price levels.
    
    Args:
        order_book: Order book data from get_order_book function
        levels: Number of price levels to display
    """
    if order_book is None:
        print("No order book data to plot")
        return
    
    # Limit to specified number of levels
    bids = order_book["bids"].head(levels)
    asks = order_book["asks"].head(levels)
    
    # Prepare the plot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot bids (buy orders) in green
    ax.barh(
        range(len(bids)), 
        bids["quantity"], 
        height=0.4, 
        left=0, 
        color='green', 
        alpha=0.5,
        label='Bids'
    )
    for i, (price, qty) in enumerate(zip(bids["price"], bids["quantity"])):
        ax.text(qty/2, i, f"{price:.2f} | {qty:.2f}", 
                va='center', ha='center', color='black', fontweight='bold')
    
    # Plot asks (sell orders) in red
    ax.barh(
        range(len(asks)), 
        asks["quantity"], 
        height=0.4, 
        left=0, 
        color='red', 
        alpha=0.5,
        label='Asks'
    )
    for i, (price, qty) in enumerate(zip(asks["price"], asks["quantity"])):
        ax.text(qty/2, i, f"{price:.2f} | {qty:.2f}", 
                va='center', ha='center', color='black', fontweight='bold')
    
    # Calculate mid price
    mid_price = (float(bids["price"].iloc[0]) + float(asks["price"].iloc[0])) / 2
    
    # Set labels and title
    ax.set_yticks(range(max(len(bids), len(asks))))
    ax.set_title(f"Order Book - Mid Price: {mid_price:.4f}")
    ax.set_xlabel("Quantity")
    ax.set_ylabel("Price Levels")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print additional information
    spread = float(asks["price"].iloc[0]) - float(bids["price"].iloc[0])
    spread_pct = (spread / mid_price) * 100

    print(f"Best bid: {bids['price'].iloc[0]}")
    print(f"Best ask: {asks['price'].iloc[0]}")
    print(f"Spread: {spread:.6f} ({spread_pct:.4f}%)")
    print(f"Total bid quantity (displayed): {bids['quantity'].sum()}")
    print(f"Total ask quantity (displayed): {asks['quantity'].sum()}")

In [7]:
def analyze_order_book_imbalance(order_book, levels=10):
    """
    Calculate and display order book imbalance metrics.
    
    Args:
        order_book: Order book data from get_order_book function
        levels: Number of price levels to consider
    """
    if order_book is None:
        print("No order book data to analyze")
        return
    
    # Limit to specified number of levels
    bids = order_book["bids"].head(levels)
    asks = order_book["asks"].head(levels)
    
    # Calculate total volumes
    total_bid_volume = bids["quantity"].sum()
    total_ask_volume = asks["quantity"].sum()
    
    # Calculate order book imbalance
    imbalance_ratio = (total_bid_volume - total_ask_volume) / (total_bid_volume + total_ask_volume)
    imbalance_pct = imbalance_ratio * 100
    
    print(f"Order Book Imbalance Analysis (Top {levels} levels):")
    print(f"Total bid volume: {total_bid_volume:.4f}")
    print(f"Total ask volume: {total_ask_volume:.4f}")
    print(f"Imbalance ratio: {imbalance_ratio:.4f}")
    print(f"Imbalance percentage: {imbalance_pct:.2f}%")
    
    if imbalance_ratio > 0.2:
        print("Strong buying pressure detected")
    elif imbalance_ratio < -0.2:
        print("Strong selling pressure detected")
    else:
        print("Relatively balanced order book")
    
    return imbalance_ratio